# Task 4: Context-Aware Chatbot Using LangChain & RAG (with Groq)

This notebook builds a chatbot that:
- Remembers conversation history (`ConversationBufferMemory`)
- Retrieves relevant information from a custom document corpus (vector store)
- Uses LangChain + **Groq's free LLM API** (llama-3.3-70b-versatile) to generate answers
- Uses **free HuggingFace embeddings** (no API key needed)
- Deploys with Streamlit

**Requirements:** Groq API key (free) – sign up at https://console.groq.com

## 1. Install Dependencies

In [1]:
!pip uninstall langchain langchain-community langchain-core langchain-groq -y
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers

  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_core-1.3.2-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_classic-1.0.4-py3-none-any.whl.metadata (4.8 kB)
Using cached langchain-1.2.15-py3-none-any.whl (112 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_groq-1.1.2-py3-none-any.whl (19 kB)
Using cached langchain_classic-1.0.4-py3-none-any.whl (1.0 MB)
Using cached langchain_core-1.3.2-py3-none-any.whl (542 kB)


In [3]:
!pip install langchain-text-splitters

## 2. Import Libraries & Set API Key (Groq)

In [5]:
import os
from getpass import getpass
from operator import itemgetter

from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Set Groq API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


## 3. Load Custom Corpus (Example Documents)

In [6]:
documents = [
    "Artificial Intelligence (AI) is intelligence demonstrated by machines, in contrast to natural intelligence displayed by humans and animals. AI research has been defined as the field of study of intelligent agents.",
    "Machine Learning (ML) is a subset of AI that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data and use it to learn for themselves.",
    "Deep Learning is a subset of machine learning that uses neural networks with many layers. It has driven many recent advances in computer vision, natural language processing, and speech recognition.",
    "LangChain is a framework for developing applications powered by language models. It provides tools for chaining LLM calls, managing memory, and integrating external data sources.",
    "Retrieval-Augmented Generation (RAG) is a technique that combines retrieval of relevant documents with generative LLMs. It helps reduce hallucinations and ground answers in external knowledge.",
    "Streamlit is an open-source Python library that makes it easy to create custom web apps for machine learning and data science. You can build interactive chatbots in just a few lines of code."
]

with open("custom_corpus.txt", "w") as f:
    f.write("\n\n".join(documents))

loader = TextLoader("custom_corpus.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

Created 8 chunks


## 4. Split Documents into Chunks

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("✅ Vector store ready")

/tmp/ipykernel_17342/3995081503.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector store ready


## 5. Create Vector Store

We use `llama-3.3-70b-versatile`

In [11]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the provided context to answer the user's question. If you don't know, just say so. Keep answers concise.\n\nContext: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL chain: retrieve -> format -> prompt -> llm
rag_chain = (
    {
        "context": itemgetter("input") | retriever | format_docs,
        "chat_history": itemgetter("chat_history"),
        "input": itemgetter("input"),
    }
    | prompt
    | llm
)
print("✅ RAG chain created")

✅ RAG chain created


## 6. Set Up LLM (Groq - Free) and Conversation Memory

In [12]:
session_store = {}

def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

conversational_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)
print("✅ Conversation memory added")

✅ Conversation memory added


## 7. Create RAG Chain with Chat History

In [13]:
res1 = conversational_chain.invoke(
    {"input": "What is Artificial Intelligence?"},
    config={"configurable": {"session_id": "test_user"}}
)
print("User: What is Artificial Intelligence?")
print(f"Bot: {res1.content}\n")

# Follow-up question
res2 = conversational_chain.invoke(
    {"input": "How does it relate to Machine Learning?"},
    config={"configurable": {"session_id": "test_user"}}
)
print("User: How does it relate to Machine Learning?")
print(f"Bot: {res2.content}")

User: What is Artificial Intelligence?
Bot: Artificial Intelligence (AI) is intelligence demonstrated by machines, in contrast to natural intelligence displayed by humans and animals.

User: How does it relate to Machine Learning?
Bot: Machine Learning (ML) is a subset of Artificial Intelligence (AI) that enables systems to learn and improve from experience without being explicitly programmed.


## 8. Test the Chatbot (In Notebook)

In [14]:
%%writefile app.py
import streamlit as st
from operator import itemgetter
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
import os

# Set API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = st.secrets.get("GROQ_API_KEY", "")

st.title("📚 Context-Aware Chatbot with RAG (Groq + Memory)")

@st.cache_resource
def load_retriever():
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
    return vector_store.as_retriever(search_kwargs={"k": 3})

retriever = load_retriever()
llm = ChatGroq(model="mixtral-8x7b-32768", temperature=0.2)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the context to answer. Keep concise.\n\nContext: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": itemgetter("input") | retriever | format_docs,
        "chat_history": itemgetter("chat_history"),
        "input": itemgetter("input"),
    }
    | prompt
    | llm
)

store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if user_input := st.chat_input():
    st.chat_message("user").write(user_input)
    with st.spinner("Thinking..."):
        response = conversational_chain.invoke(
            {"input": user_input},
            config={"configurable": {"session_id": "streamlit_user"}}
        )
        answer = response.content
    st.chat_message("assistant").write(answer)
    st.session_state.messages.append({"role": "user", "content": user_input})
    st.session_state.messages.append({"role": "assistant", "content": answer})

Writing app.py


## 9. Deploy with Streamlit (Using Groq & HuggingFace Embeddings)

In [15]:
print("""
✅ Streamlit app saved as 'app.py'

To run the chatbot web app:
1. In your terminal, navigate to this folder
2. Run: streamlit run app.py

If you haven't set the GROQ_API_KEY environment variable, you can:
- Create a file .streamlit/secrets.toml with: GROQ_API_KEY = "your-key"
- Or set it in terminal: export GROQ_API_KEY="your-key" (Linux/Mac) or set GROQ_API_KEY=your-key (Windows)

The app will open in your browser.
""")


✅ Streamlit app saved as 'app.py'

To run the chatbot web app:
1. In your terminal, navigate to this folder
2. Run: streamlit run app.py

If you haven't set the GROQ_API_KEY environment variable, you can:
- Create a file .streamlit/secrets.toml with: GROQ_API_KEY = "your-key"
- Or set it in terminal: export GROQ_API_KEY="your-key" (Linux/Mac) or set GROQ_API_KEY=your-key (Windows)

The app will open in your browser.



## Summary

I built a **completely free** RAG chatbot using:
- Groq's free LLM API (llama-3.3-70b-versatile)
- HuggingFace embeddings (local, no key)
- Chroma vector store
- Conversation memory
- Streamlit deployment